# 01 — EDA

Bivariate-first: every section compares a feature *against the target* rather than
cataloguing columns in isolation. That is what makes an hour of looking productive —
you are hunting for signal, not producing a data dictionary.

The analysis lives in [`eda.py`](eda.py); the cells below call it and
show the output. Read a function's docstring for *why* a step exists — this notebook
says what to look at and what the answer means.

**Your output from this notebook is the last section**: the `CATEGORICAL` /
`NUMERIC` / `ORDINAL` / `EXCLUDED` lists that feed feature selection in `../src/run.py`.

In [ ]:
import sys

sys.path.insert(0, "../src")

import eda

eda.set_plot_theme()

## Data setup

Rows with a missing target are dropped, not imputed. An unlabelled row is not a
training example, and filling the label invents outcomes.

In [ ]:
df = eda.load()
df.info()

In [ ]:
FEATURES = eda.features(df)
print(f"{len(FEATURES)} candidate features (target and ID columns excluded)")
df.describe().T

## Target balance

The first number that changes the modelling plan. Severe imbalance decides the
metric (PR-AUC, not ROC-AUC), the sampling strategy, and whether a threshold can be
chosen at all.

**Read the positives-per-fold number before trusting any metric below.** If it is in
the single digits, cross-validated scores carry fold-to-fold variance far larger than
any model difference you are likely to see. Report the spread, not just the mean.

In [ ]:
balance = eda.target_balance(df)

## Leakage audit — before looking at correlations

The highest-value cell here, and the one most often skipped. A leaked feature does
not raise an error: it returns a great score and reads as success.

For every column at the top of this ranking, ask: **was this value knowable at the
moment the prediction would be made?** Anything recorded *because* the outcome
happened is leakage, however predictive it looks.

In the 2020 original, `has_paid` showed a strong association with the non-default
group. For a default model that is exactly the shape a leak takes — "did they pay"
may be the outcome restated rather than a predictor.

In [ ]:
leaks = eda.leak_scan(df)

Record the verdict explicitly. A column reviewed and kept is different from a
column nobody looked at — this list is the difference.

In [ ]:
SUSPECTED_LEAKS: list[str] = [
    # "has_paid",       # post-outcome? confirm before including
    # "recovery_debt",  # recovery implies a default already occurred
]
print(f"excluded as leakage: {SUSPECTED_LEAKS or 'none — but the scan above was reviewed'}")

## Missingness — share and pattern

Per-column rates decide impute-vs-drop. **Co-occurrence** decides whether missingness
is itself a signal: columns that go missing together usually mean a whole record type
is absent, and an indicator captures that where a fill value destroys it.

In [ ]:
miss = eda.missingness(df)

In [ ]:
fig_miss = eda.plot_missingness_pattern(df)

## Feature types

Inferred with the same code the pipeline uses (`ml.transform.columns`), so the
notebook and `run.py` cannot drift apart in their view of the columns.

Inference is a starting point, **corrected by hand**: an ID read as int64 becomes a
numeric feature silently, and a 0–4 status code read as a float becomes a magnitude
when it is really a rank. That correction lands in `ORDINAL` at the handoff.

In [ ]:
plan = eda.column_plan(df)

Cardinality decides the encoder: low → one-hot, high → target encoding or rare-level collapse.

In [ ]:
card = eda.cardinality(df, plan)

## Bivariate — categorical features vs target

Default rate by level against the base rate (dashed line). More readable than two
overlaid histograms when the classes are this imbalanced — at a ~1.75% positive rate
the default group is invisible on a shared count axis. A level whose bar sits far
from the line is where the signal is.

In [ ]:
fig_cat = eda.plot_categorical_vs_target(df, plan)

In [ ]:
eda.high_cardinality_levels(df, plan)

## Univariate — numeric distributions

Skew and outliers decide scaling and whether a transform is needed.

In [ ]:
NUM = eda.numeric_features(plan, exclude=SUSPECTED_LEAKS)
print(f"{len(NUM)} numeric features (leak candidates excluded)")

In [ ]:
fig_dist = eda.plot_numeric_distributions(df, NUM)

In [ ]:
skew = eda.skewness(df, NUM)

## Bivariate — numeric features vs target

Violin plots by target class. With few positives the positive-side violin is thin —
read it as indicative, not conclusive.

In [ ]:
fig_biv = eda.plot_numeric_vs_target(df, NUM)

## Correlation between features

Clusters mean redundancy: keep the most interpretable member of each cluster rather
than all of them. In the 2020 original the `status_*` features were mutually
correlated and dropped out in feature selection.

In [ ]:
fig_corr = eda.plot_correlation(df, NUM)

In [ ]:
pairs = eda.redundant_pairs(df, NUM)

---
## Handoff — your output from this notebook

Fill these in from what you found above. They are the input to feature selection and
the training config. A column in none of these lists is silently dropped, so the
counts have to reconcile — `reconcile` names anything you did not decide about.

On a fresh run every feature is unaccounted. That is the gate working, not a bug.

In [ ]:
CATEGORICAL: list[str] = [
    # from `plan.categorical`, minus anything excluded above
]

NUMERIC: list[str] = [
    # from `plan.numeric`, minus SUSPECTED_LEAKS and redundant pairs
]

ORDINAL: list[str] = [
    # status_*-style rank codes that inference read as numeric — these need
    # ordinal treatment, not magnitude
]

EXCLUDED: dict[str, str] = {
    # "column": "why — leakage / redundant / no variance"
}

unaccounted = eda.reconcile(FEATURES, CATEGORICAL, NUMERIC, ORDINAL, EXCLUDED)

### Feature engineering — brainstorm before building

Before writing a single transform: what do you know about this domain that the data
does not say? Ratios that matter, thresholds that are meaningful, time windows that
should be compared rather than used raw.

This frame has features aggregated over *overlapping* windows (0–12m, 12–24m, 0–24m).
Overlapping windows are both an opportunity (a 0–12m vs 12–24m ratio is a trend
signal) and a trap (0–24m is not independent of either). Write the candidates down
before implementing them:

1.
2.
3.